In [43]:
from pathlib import Path
ROOT = Path().resolve().parent  # Scripts/ → repo root

BASE_PATH = ROOT / "Model_Outputs"

DATA_PATH           = ROOT / "Data_Files"   / "stroke_master_final_cleaned.parquet"
COLS_TO_DROP_PATH   = ROOT / "Stat_Outputs" / "cols_to_drop.csv"
NUM_FEAT_PATH       = ROOT / "Stat_Outputs" / "num_feat.csv"
REMOVED_FEATURES_PATH = ROOT / "Stat_Outputs" / "removed_features.csv"

In [44]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.base import clone

from sklearn.metrics import (
    matthews_corrcoef,
    accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    make_scorer
)

from sklearn.linear_model import LogisticRegression  
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

from xgboost import XGBClassifier


# ----------------------------
# Config helpers
# ----------------------------
def get_thresh_grid(n=99):
    return np.linspace(0.01, 0.99, n)


def ensure_outdir(outdir):
    outdir = os.path.abspath(outdir)
    os.makedirs(outdir, exist_ok=True)
    return outdir


# ----------------------------
# Metric helpers
# ----------------------------
def metric_name_normalize(metric):
    metric = str(metric).strip().lower()
    if metric in {"mcc"}:
        return "mcc"
    if metric in {"macro_f1", "macro-f1", "macrof1"}:
        return "macro_f1"
    if metric in {"f1_0", "f10", "f1-0"}:
        return "f1_0"
    if metric in {"f1_1", "f11", "f1-1"}:
        return "f1_1"
    raise ValueError("metric must be 'mcc', 'macro_f1', 'f1_0', or 'f1_1'.")


def _safe_average_precision(y_true_bin, p_pos):
    """
    Returning average precision safely.
    If y_true has only one class present, AP is not meaningful; return np.nan.
    """
    y_true_bin = np.asarray(y_true_bin).astype(int)
    if len(np.unique(y_true_bin)) < 2:
        return np.nan
    return float(average_precision_score(y_true_bin, p_pos))


def score_at_threshold(y_true, p, threshold):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)
    yhat = (p >= float(threshold)).astype(int)

    cm = confusion_matrix(y_true, yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel().tolist()

    # F1 per class
    f1_1 = float(f1_score(y_true, yhat, pos_label=1, average="binary", zero_division=0))
    f1_0 = float(f1_score(y_true, yhat, pos_label=0, average="binary", zero_division=0))
    f1_gap = float(abs(f1_0 - f1_1))

    # PR-AUC per class
    pr_auc_1 = _safe_average_precision((y_true == 1).astype(int), p)
    pr_auc_0 = _safe_average_precision((y_true == 0).astype(int), 1.0 - p)

    # Macro PR-AUC
    if np.isnan(pr_auc_0) and np.isnan(pr_auc_1):
        pr_auc_macro = np.nan
    elif np.isnan(pr_auc_0):
        pr_auc_macro = pr_auc_1
    elif np.isnan(pr_auc_1):
        pr_auc_macro = pr_auc_0
    else:
        pr_auc_macro = float((pr_auc_0 + pr_auc_1) / 2.0)

    out = {
        "threshold": float(threshold),

        "accuracy": float(accuracy_score(y_true, yhat)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, yhat)),

        "precision": float(precision_score(y_true, yhat, zero_division=0)),
        "recall": float(recall_score(y_true, yhat, zero_division=0)),
        "f1": float(f1_1),

        "f1_0": f1_0,
        "f1_1": f1_1,
        "abs_f1_gap": f1_gap,
        "macro_f1": float(f1_score(y_true, yhat, average="macro", zero_division=0)),

        "mcc": float(matthews_corrcoef(y_true, yhat)),

        "roc_auc": float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) == 2 else np.nan,

        # Keep your old column name as class-1 PR-AUC for backward compatibility
        "pr_auc": pr_auc_1,

        # New fields
        "pr_auc_0": pr_auc_0,
        "pr_auc_1": pr_auc_1,
        "pr_auc_macro": pr_auc_macro,

        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }
    return out


def get_prob_or_score(model, X_in):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_in)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X_in)
        s = (s - np.min(s)) / (np.max(s) - np.min(s) + 1e-12)
        return s
    raise ValueError("Model is lacking predict_proba and decision_function.")


def pick_threshold(y_true, p, metric, thresh_grid):
    metric = metric_name_normalize(metric)

    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)

    best_t = 0.5
    best_val = -np.inf

    for t in thresh_grid:
        yhat = (p >= float(t)).astype(int)
        if len(np.unique(yhat)) < 2:
            continue

        if metric == "mcc":
            val = matthews_corrcoef(y_true, yhat)
        elif metric == "macro_f1":
            val = f1_score(y_true, yhat, average="macro", zero_division=0)
        elif metric == "f1_0":
            val = f1_score(y_true, yhat, pos_label=0, average="binary", zero_division=0)
        elif metric == "f1_1":
            val = f1_score(y_true, yhat, pos_label=1, average="binary", zero_division=0)
        else:
            raise ValueError("Unsupported metric in pick_threshold")

        if val > best_val:
            best_val = val
            best_t = float(t)

    return float(best_t), float(best_val)


def prefix_dict(d, prefix):
    return {f"{prefix}_{k}": v for k, v in d.items()}



from sklearn.model_selection import RandomizedSearchCV

def make_search(estimator, model_name, param_grids, scorer, cv, n_jobs, random_state, verbose):
    if model_name == "XGBoost":
        # RandomizedSearch for XGB because the space is large
        return RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_grids[model_name],
            n_iter=60,                 # try 60 first; increase to 120 if needed
            scoring=scorer,
            cv=cv,
            n_jobs=n_jobs,
            random_state=random_state,
            verbose=verbose,
            refit=True,
            error_score="raise",
        )
    else:
        return GridSearchCV(
            estimator=estimator,
            param_grid=param_grids[model_name],
            scoring=scorer,
            cv=cv,
            n_jobs=n_jobs,
            verbose=verbose,
            refit=True,
            error_score="raise",
        )


# ----------------------------
# Data loading (NO IMPUTATION)
# ----------------------------
def load_complete_case_dataset(
    data_path,
    cols_to_drop_path,
    num_feat_path,
    removed_features_path,
    target_col="EMSSceneTimeMin",
    key_col="PcrKey",
    y_col="scene_time_binary",
    threshold_minutes=10,
):
    df = pd.read_parquet(data_path)

    cols_to_drop = pd.read_csv(cols_to_drop_path)["column_name"].astype(str).tolist()
    num_feat_original = pd.read_csv(num_feat_path)["column_name"].astype(str).tolist()
    removed = pd.read_csv(removed_features_path).iloc[:, 0].astype(str).tolist()

    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")
    df = df.drop(columns=[c for c in removed if c in df.columns], errors="ignore")

    num_feat = [c for c in num_feat_original if c in df.columns and c != target_col]
    cols_to_exclude = set(num_feat) | {key_col, target_col}
    cat_feat = [c for c in df.columns if c not in cols_to_exclude]

    for c in num_feat + [target_col]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=[target_col]).copy()
    df[y_col] = (df[target_col] > float(threshold_minutes)).astype(int)

    # Converting pandas NA to numpy nan to avoid sklearn mask errors
    df = df.replace({pd.NA: np.nan})

    feature_cols = cat_feat + num_feat
    df_cc = df.dropna(subset=feature_cols + [y_col]).copy()

    X = df_cc[feature_cols].copy()
    y = df_cc[y_col].astype(int).copy()

    meta = {
        "cat_feat": cat_feat,
        "num_feat": num_feat,
        "feature_cols": feature_cols,
        "target_col": target_col,
        "y_col": y_col,
    }
    return X, y, meta


def build_preprocessor(cat_feat, num_feat):
    prep = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feat),
            ("num", StandardScaler(), num_feat),
        ],
        remainder="drop",
    )
    return prep



import xgboost as xgb

def add_xgb_backend_kwargs(xgb_kwargs: dict, use_gpu_for_xgb: bool) -> dict:
    """
    Making XGBoost kwargs safe across local CPU and Colab GPU.
    """
    xgb_kwargs = dict(xgb_kwargs)

    if not use_gpu_for_xgb:
        xgb_kwargs.update({"tree_method": "hist"})
        for k in ["device", "predictor", "gpu_id"]:
            xgb_kwargs.pop(k, None)
        return xgb_kwargs

    # Try GPU, fallback to CPU
    try:
        xgb_kwargs.update({"tree_method": "gpu_hist"})
        ver = getattr(xgb, "__version__", "")
        major = int(ver.split(".")[0]) if ver and ver[0].isdigit() else 1

        if major >= 2:
            xgb_kwargs.update({"device": "cuda"})
            xgb_kwargs.pop("gpu_id", None)
            xgb_kwargs.pop("predictor", None)
        else:
            xgb_kwargs.update({"gpu_id": 0})
            xgb_kwargs.pop("device", None)
            xgb_kwargs.pop("predictor", None)

        return xgb_kwargs
    except Exception:
        xgb_kwargs.update({"tree_method": "hist"})
        for k in ["device", "predictor", "gpu_id"]:
            xgb_kwargs.pop(k, None)
        return xgb_kwargs


# ----------------------------
# Models, workflows, grids
# ----------------------------
def build_base_estimators(random_state, use_gpu_for_xgb=False, xgb_n_jobs=1):
    xgb_kwargs = {
        "random_state": random_state,
        "n_jobs": int(xgb_n_jobs),   # set 1 when GridSearchCV is parallel
        "eval_metric": "logloss",
    }
    xgb_kwargs = add_xgb_backend_kwargs(xgb_kwargs, use_gpu_for_xgb=use_gpu_for_xgb)

    return {
        "Logistic Regression": LogisticRegression(
            random_state=random_state, max_iter=2000, solver="lbfgs", n_jobs=-1
        ),
        "Decision Tree": DecisionTreeClassifier(random_state=random_state),
        "Random Forest": RandomForestClassifier(random_state=random_state, n_jobs=-1),
        "XGBoost": XGBClassifier(**xgb_kwargs),
    }


def build_weighted_estimators(y_train, random_state, use_gpu_for_xgb=False, xgb_n_jobs=1):
    classes = np.array([0, 1])
    cw = compute_class_weight(class_weight="balanced", classes=classes, y=np.asarray(y_train))
    class_weight = {0: float(cw[0]), 1: float(cw[1])}

    n_pos = int((np.asarray(y_train) == 1).sum())
    n_neg = int((np.asarray(y_train) == 0).sum())
    scale_pos_weight = float(n_neg / max(n_pos, 1))

    xgb_kwargs = {
        "random_state": random_state,
        "n_jobs": int(xgb_n_jobs),
        "eval_metric": "logloss",
        "scale_pos_weight": scale_pos_weight,
    }
    xgb_kwargs = add_xgb_backend_kwargs(xgb_kwargs, use_gpu_for_xgb=use_gpu_for_xgb)

    return {
        "Logistic Regression": LogisticRegression(
            random_state=random_state, max_iter=2000, solver="lbfgs",
            n_jobs=-1, class_weight=class_weight
        ),
        "Decision Tree": DecisionTreeClassifier(
            random_state=random_state, class_weight=class_weight
        ),
        "Random Forest": RandomForestClassifier(
            random_state=random_state, n_jobs=-1, class_weight=class_weight
        ),
        "XGBoost": XGBClassifier(**xgb_kwargs),
    }


def build_pipeline(workflow, prep, clf, random_state):
    if workflow in {"unbalanced", "weighted"}:
        return SkPipeline([("prep", prep), ("clf", clf)])

    if workflow == "undersampled":
        return ImbPipeline([
            ("prep", prep),
            ("rus", RandomUnderSampler(random_state=random_state)),
            ("clf", clf),
        ])

    raise ValueError("workflow must be: unbalanced, weighted, undersampled.")


def get_param_grids():
    return {
        "Logistic Regression": {"clf__C": [0.01, 0.1, 1, 10]},
        "Decision Tree": {
            "clf__max_depth": [5, 10, 15, None],
            "clf__min_samples_split": [2, 5, 10],
        },
        "Random Forest": {
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [10, 15, 20],
        },
        "XGBoost": {
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [3, 6, 10],
            "clf__learning_rate": [0.01, 0.1, 0.2],
        },
    }


def get_param_grids_v2():
    return {
        "Logistic Regression": {
            "clf__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10],
        },

        "Decision Tree": {
            "clf__max_depth": [3, 5, 7, 10, 15, None],
            "clf__min_samples_split": [2, 5, 10, 20, 50],
            "clf__min_samples_leaf": [1, 2, 5, 10, 20],
            "clf__ccp_alpha": [0.0, 1e-4, 5e-4, 1e-3, 5e-3],
        },

        "Random Forest": {
            "clf__n_estimators": [300, 600, 900],
            "clf__max_depth": [6, 10, 15, 20, None],
            "clf__min_samples_split": [2, 5, 10, 20],
            "clf__min_samples_leaf": [1, 2, 5, 10],
            "clf__max_features": ["sqrt", "log2", 0.3, 0.5, 0.8],
            "clf__class_weight": [None],  # keep RF consistent; use "weighted" workflow for weighting
        },

        "XGBoost": {
            "clf__n_estimators": [300, 600, 900, 1200],
            "clf__learning_rate": [0.01, 0.03, 0.05, 0.1],
            "clf__max_depth": [2, 3, 4, 6, 8],
            "clf__min_child_weight": [1, 2, 5, 10, 20],
            "clf__gamma": [0, 0.5, 1, 2, 5],
            "clf__subsample": [0.6, 0.8, 1.0],
            "clf__colsample_bytree": [0.6, 0.8, 1.0],
            "clf__reg_alpha": [0.0, 1e-3, 1e-2, 1e-1, 1.0],
            "clf__reg_lambda": [0.5, 1.0, 2.0, 5.0, 10.0],
        },
    }




def get_cv_scorer(metric):
    metric = metric_name_normalize(metric)
    if metric == "mcc":
        return make_scorer(matthews_corrcoef)
    if metric == "macro_f1":
        return make_scorer(f1_score, average="macro", zero_division=0)
    if metric == "f1_0":
        return make_scorer(f1_score, pos_label=0, average="binary", zero_division=0)
    if metric == "f1_1":
        return make_scorer(f1_score, pos_label=1, average="binary", zero_division=0)
    raise ValueError("metric must be one of: mcc, macro_f1, f1_0, f1_1")


# ----------------------------
# Candidate selection
# ----------------------------
def select_best_by_valid_generalization(cand_df, metric, eps):
    metric = metric_name_normalize(metric)
    valid_key = f"valid_tuned_{metric}"

    best_valid = float(cand_df[valid_key].max())
    pool = cand_df[cand_df[valid_key] >= (best_valid - float(eps))].copy()
    if pool.empty:
        pool = cand_df.copy()

    best_row = pool.sort_values(
        by=[
            "gen_gap_metric",
            valid_key,
            "valid_tuned_pr_auc_macro",
            "best_cv_metric",
        ],
        ascending=[True, False, False, False],
    ).iloc[0]

    return best_row


# ----------------------------
# OOF evaluation for joint tuning
# ----------------------------
def oof_eval_for_params(pipe, X_train, y_train, cv, metric, thresh_grid):
    metric = metric_name_normalize(metric)

    y_arr = np.asarray(y_train).astype(int)
    oof_p = np.full(shape=(len(y_arr),), fill_value=np.nan, dtype=float)

    for tr_idx, va_idx in cv.split(X_train, y_arr):
        X_tr = X_train.iloc[tr_idx]
        y_tr = y_arr[tr_idx]
        X_va = X_train.iloc[va_idx]

        pipe_fold = clone(pipe)
        pipe_fold.fit(X_tr, y_tr)
        p_va = get_prob_or_score(pipe_fold, X_va)
        oof_p[va_idx] = p_va

    if np.any(np.isnan(oof_p)):
        raise RuntimeError("OOF probabilities are containing NaNs.")

    best_t, best_metric_val = pick_threshold(y_arr, oof_p, metric=metric, thresh_grid=thresh_grid)
    metrics_at_t = score_at_threshold(y_arr, oof_p, threshold=best_t)

    return best_t, best_metric_val, metrics_at_t

In [45]:
def run_tvt_pipeline(
    metric="mcc",
    eps=0.02,
    random_state=42,
    n_splits=5,
    thresh_grid=None,
    outdir=None,
    use_gpu_for_xgb=True,
):
    metric = metric_name_normalize(metric)
    if thresh_grid is None:
        thresh_grid = get_thresh_grid(99)
    if outdir is None:
        outdir = os.path.join(BASE_PATH, f"out_tvt_{metric}")
    ensure_outdir(outdir)

    # Loading complete-case cohort (no imputation)
    X, y, meta = load_complete_case_dataset(
        data_path=DATA_PATH,
        cols_to_drop_path=COLS_TO_DROP_PATH,
        num_feat_path=NUM_FEAT_PATH,
        removed_features_path=REMOVED_FEATURES_PATH,
        target_col="EMSSceneTimeMin",
        key_col="PcrKey",
        y_col="scene_time_binary",
        threshold_minutes=10,
    )

    print("Complete-case cohort:", X.shape, "positive rate:", round(float(y.mean()), 6))

    # Splitting 80/10/10
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=random_state
    )
    X_valid, X_test, y_valid, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=random_state
    )

    # Saving split indices
    np.save(os.path.join(outdir, "idx_train.npy"), X_train.index.to_numpy())
    np.save(os.path.join(outdir, "idx_valid.npy"), X_valid.index.to_numpy())
    np.save(os.path.join(outdir, "idx_test.npy"), X_test.index.to_numpy())

    print("Train:", X_train.shape, "Valid:", X_valid.shape, "Test:", X_test.shape)

    # Building preprocessor
    prep = build_preprocessor(meta["cat_feat"], meta["num_feat"])

    # Building models
    base_models = build_base_estimators(random_state, use_gpu_for_xgb=use_gpu_for_xgb)
    weighted_models = build_weighted_estimators(y_train, random_state, use_gpu_for_xgb=use_gpu_for_xgb)

    param_grids = get_param_grids()
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scorer = get_cv_scorer(metric)

    workflows = [
        ("unbalanced", base_models),
        ("weighted", weighted_models),
        ("undersampled", base_models),
    ]

    candidates = []

    for workflow_name, model_dict in workflows:
        for model_name, clf in model_dict.items():
            pipe = build_pipeline(workflow_name, prep, clf, random_state)

            grid = GridSearchCV(
                estimator=pipe,
                param_grid=param_grids[model_name],
                scoring=scorer,
                cv=cv,
                n_jobs=-1,
                verbose=1,
                error_score="raise",
            )

            grid.fit(X_train, y_train)
            best_est = grid.best_estimator_

            # Tuning threshold on VALID only
            p_valid = get_prob_or_score(best_est, X_valid)
            best_t, best_valid_metric_val = pick_threshold(
                y_true=y_valid.to_numpy(),
                p=p_valid,
                metric=metric,
                thresh_grid=thresh_grid,
            )

            # Scoring TRAIN and VALID at tuned threshold
            p_train = get_prob_or_score(best_est, X_train)
            train_tuned = score_at_threshold(y_train, p_train, threshold=best_t)
            valid_tuned = score_at_threshold(y_valid, p_valid, threshold=best_t)

            gen_gap_metric = float(abs(train_tuned[metric] - valid_tuned[metric]))
            gen_gap_pr_auc = float(abs(train_tuned["pr_auc_macro"] - valid_tuned["pr_auc_macro"]))

            cand = {
                "workflow": workflow_name,
                "model": model_name,
                "best_params": grid.best_params_,
                "best_cv_metric": float(grid.best_score_),
                "best_threshold_valid": float(best_t),
                "best_valid_metric_val": float(best_valid_metric_val),
                "gen_gap_metric": gen_gap_metric,
                "gen_gap_pr_auc": gen_gap_pr_auc,
                **prefix_dict(train_tuned, "train_tuned"),
                **prefix_dict(valid_tuned, "valid_tuned"),
                "_best_estimator": best_est,
            }
            cand["candidate_id"] = len(candidates)
            candidates.append(cand)

            #print('\n')
            print(
                "Done:", workflow_name, "|", model_name,
                "| best_cv_metric:", round(float(grid.best_score_), 6),
                "| best_t:", round(best_t, 4),
                "| valid", metric, ":", round(float(valid_tuned[metric]), 6),
                "| valid f1_0:", round(float(valid_tuned["f1_0"]), 6),
                "| valid f1_1:", round(float(valid_tuned["f1_1"]), 6),
                "| abs(f1_0-f1_1):", round(float(valid_tuned["abs_f1_gap"]), 6),
                "| valid pr_auc_0:", round(float(valid_tuned["pr_auc_0"]), 6),
                "| valid pr_auc_1:", round(float(valid_tuned["pr_auc_1"]), 6),
                "| valid pr_auc_macro:", round(float(valid_tuned["pr_auc_macro"]), 6),
                "| metric_gap(train-valid):", round(float(gen_gap_metric), 6),
            )

    # Building candidates dataframe (dropping estimator object)
    cand_df = pd.DataFrame([{k: v for k, v in c.items() if not k.startswith("_")} for c in candidates])

    candidates_csv = os.path.join(outdir, "candidates_valid_only.csv")
    cand_df.to_csv(candidates_csv, index=False)
    print("Saved:", candidates_csv)

    # Selecting best using epsilon floor on VALID metric then generalization gap
    best_row = select_best_by_valid_generalization(cand_df, metric=metric, eps=eps)
    best_id = int(best_row["candidate_id"])
    best_est = candidates[best_id]["_best_estimator"]
    best_t = float(candidates[best_id]["best_threshold_valid"])

    print("Selected best:")
    print("  workflow:", best_row["workflow"])
    print("  model   :", best_row["model"])
    print("  metric  :", metric)
    print("  valid metric:", round(float(best_row[f"valid_tuned_{metric}"]), 6))
    print("  gap:", round(float(best_row["gen_gap_metric"]), 6))
    print("  best_t:", round(best_t, 6))

    # Final TEST evaluation (single model only)
    p_test = get_prob_or_score(best_est, X_test)
    test_default = score_at_threshold(y_test, p_test, threshold=0.5)
    test_tuned = score_at_threshold(y_test, p_test, threshold=best_t)

    final = best_row.to_dict()
    final.update(prefix_dict(test_default, "test_default"))
    final.update(prefix_dict(test_tuned, "test_tuned"))
    final["selected_metric"] = metric
    final["eps"] = float(eps)

    best_csv = os.path.join(outdir, "best_model_final_test.csv")
    pd.DataFrame([final]).to_csv(best_csv, index=False)
    print("Saved:", best_csv)

    return (
    cand_df,
    pd.DataFrame([final]),
    best_est,                       # fitted pipeline
    float(best_t),                  # tuned threshold
    {
        "train_idx": X_train.index.to_numpy(),
        "valid_idx": X_valid.index.to_numpy(),
        "test_idx":  X_test.index.to_numpy(),
    },
    )


# Example runs:
# cand_df, best_df = run_tvt_pipeline(metric="mcc", eps=0.02)
# cand_df, best_df = run_tvt_pipeline(metric="macro_f1", eps=0.01)



In [46]:
def run_oof_80_20_pipeline(
    metric="mcc",
    random_state=42,
    n_splits=5,
    thresh_grid=None,
    outdir=None,
    use_gpu_for_xgb=True,
):
    metric = metric_name_normalize(metric)
    if thresh_grid is None:
        thresh_grid = get_thresh_grid(99)
    if outdir is None:
        outdir = os.path.join(BASE_PATH, f"out_oof_80_20_{metric}")
    ensure_outdir(outdir)

    # Loading complete-case cohort (no imputation)
    X, y, meta = load_complete_case_dataset(
        data_path=DATA_PATH,
        cols_to_drop_path=COLS_TO_DROP_PATH,
        num_feat_path=NUM_FEAT_PATH,
        removed_features_path=REMOVED_FEATURES_PATH,
        target_col="EMSSceneTimeMin",
        key_col="PcrKey",
        y_col="scene_time_binary",
        threshold_minutes=10,
    )

    print("Complete-case cohort:", X.shape, "positive rate:", round(float(y.mean()), 6))

    # Splitting 80/20
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=random_state
    )

    np.save(os.path.join(outdir, "idx_train_80_20.npy"), X_train.index.to_numpy())
    np.save(os.path.join(outdir, "idx_test_80_20.npy"), X_test.index.to_numpy())

    print("Train:", X_train.shape, "Test:", X_test.shape)

    # Building preprocessor
    prep = build_preprocessor(meta["cat_feat"], meta["num_feat"])

    # Building models
    base_models = build_base_estimators(random_state, use_gpu_for_xgb=use_gpu_for_xgb)
    weighted_models = build_weighted_estimators(y_train, random_state, use_gpu_for_xgb=use_gpu_for_xgb)

    param_grids = get_param_grids()
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    workflows = [
        ("unbalanced", base_models),
        ("weighted", weighted_models),
        ("undersampled", base_models),
    ]

    candidates = []

    for workflow_name, model_dict in workflows:
        for model_name, clf in model_dict.items():
            base_pipe = build_pipeline(workflow_name, prep, clf, random_state)

            best_overall = None

            for params in ParameterGrid(param_grids[model_name]):
                pipe = base_pipe.set_params(**params)

                best_t, best_metric_val, oof_metrics = oof_eval_for_params(
                    pipe=pipe,
                    X_train=X_train,
                    y_train=y_train,
                    cv=cv,
                    metric=metric,
                    thresh_grid=thresh_grid,
                )

                record = {
                    "workflow": workflow_name,
                    "model": model_name,
                    "params": params,
                    "oof_best_threshold": float(best_t),
                    "oof_best_metric_val": float(best_metric_val),
                    **prefix_dict(oof_metrics, "oof"),
                }

                if best_overall is None:
                    best_overall = record
                else:
                    if record[f"oof_{metric}"] > best_overall[f"oof_{metric}"]:
                        best_overall = record

            candidates.append(best_overall)

            print(
                "Done:", workflow_name, "|", model_name,
                "| oof", metric, ":", round(float(best_overall[f"oof_{metric}"]), 6),
                "| oof f1_0:", round(float(best_overall["oof_f1_0"]), 6),
                "| oof f1_1:", round(float(best_overall["oof_f1_1"]), 6),
                "| abs(f1_0-f1_1):", round(float(best_overall["oof_abs_f1_gap"]), 6),

                # NEW: PR-AUC per class + macro
                "| oof pr_auc_0:", round(float(best_overall["oof_pr_auc_0"]), 6) if pd.notna(best_overall["oof_pr_auc_0"]) else np.nan,
                "| oof pr_auc_1:", round(float(best_overall["oof_pr_auc_1"]), 6) if pd.notna(best_overall["oof_pr_auc_1"]) else np.nan,
                "| oof pr_auc_macro:", round(float(best_overall["oof_pr_auc_macro"]), 6) if pd.notna(best_overall["oof_pr_auc_macro"]) else np.nan,

                "| best_t:", round(float(best_overall["oof_best_threshold"]), 4),
            )

    cand_df = pd.DataFrame(candidates)

    candidates_csv = os.path.join(outdir, "candidates_train_only_oof.csv")
    cand_df.to_csv(candidates_csv, index=False)
    print("Saved:", candidates_csv)

    # Selecting best candidate by OOF metric, then PR-AUC, then MCC, then ROC-AUC
    best_row = (
        cand_df.sort_values(
            by=[f"oof_{metric}", "oof_pr_auc_macro", "oof_mcc", "oof_roc_auc"],
            ascending=[False, False, False, False],
        )
        .iloc[0]
    )

    best_workflow = str(best_row["workflow"])
    best_model = str(best_row["model"])
    best_params = best_row["params"]
    best_t = float(best_row["oof_best_threshold"])

    print("Selected best:")
    print("  workflow:", best_workflow)
    print("  model   :", best_model)
    print("  metric  :", metric)
    print("  oof metric:", round(float(best_row[f"oof_{metric}"]), 6))
    print("  best_t:", round(best_t, 6))

    # Rebuilding fresh estimators for final fit
    base_models2 = build_base_estimators(random_state, use_gpu_for_xgb=use_gpu_for_xgb)
    weighted_models2 = build_weighted_estimators(y_train, random_state, use_gpu_for_xgb=use_gpu_for_xgb)

    model_lookup = {
        "unbalanced": base_models2,
        "undersampled": base_models2,
        "weighted": weighted_models2,
    }

    clf_final = model_lookup[best_workflow][best_model]
    pipe_final = build_pipeline(best_workflow, prep, clf_final, random_state)
    pipe_final = pipe_final.set_params(**best_params)
    pipe_final.fit(X_train, y_train)

    p_test = get_prob_or_score(pipe_final, X_test)
    test_default = score_at_threshold(y_test, p_test, threshold=0.5)
    test_tuned = score_at_threshold(y_test, p_test, threshold=best_t)

    final = best_row.to_dict()
    final.update(prefix_dict(test_default, "test_default"))
    final.update(prefix_dict(test_tuned, "test_tuned"))
    final["selected_metric"] = metric

    best_csv = os.path.join(outdir, "best_model_final_test_80_20.csv")
    pd.DataFrame([final]).to_csv(best_csv, index=False)
    print("Saved:", best_csv)

    return (
    cand_df,
    pd.DataFrame([final]),
    pipe_final,                     # fitted pipeline
    float(best_t),                  # tuned threshold
    {
        "train_idx": X_train.index.to_numpy(),
        "test_idx":  X_test.index.to_numpy(),
    },
    )


# Example runs:
# cand_df_oof, best_df_oof = run_oof_80_20_pipeline(metric="mcc")
# cand_df_oof, best_df_oof = run_oof_80_20_pipeline(metric="macro_f1")



In [47]:
cand_macro, best_macro, tvt_pipe, tvt_t, tvt_splits = run_tvt_pipeline(
    metric="macro_f1", eps=0.01, use_gpu_for_xgb=False
)
best_macro

Complete-case cohort: (160566, 31) positive rate: 0.868067
Train: (128452, 31) Valid: (16057, 31) Test: (16057, 31)
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Done: unbalanced | Logistic Regression | best_cv_metric: 0.593897 | best_t: 0.72 | valid macro_f1 : 0.662496 | valid f1_0: 0.417687 | valid f1_1: 0.907306 | abs(f1_0-f1_1): 0.489619 | valid pr_auc_0: 0.381955 | valid pr_auc_1: 0.956454 | valid pr_auc_macro: 0.669205 | metric_gap(train-valid): 0.059387
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Done: unbalanced | Decision Tree | best_cv_metric: 0.603953 | best_t: 0.72 | valid macro_f1 : 0.597767 | valid f1_0: 0.306757 | valid f1_1: 0.888776 | abs(f1_0-f1_1): 0.58202 | valid pr_auc_0: 0.203818 | valid pr_auc_1: 0.895326 | valid pr_auc_macro: 0.549572 | metric_gap(train-valid): 0.323317
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Done: unbalanced | Random Forest | best_cv_metric: 0.464687 | best_t: 0.85 | valid macro_f1 : 0.621519 |

,workflow,model,best_params,best_cv_metric,best_threshold_valid,best_valid_metric_val,gen_gap_metric,gen_gap_pr_auc,train_tuned_threshold,train_tuned_accuracy,...,test_tuned_pr_auc,test_tuned_pr_auc_0,test_tuned_pr_auc_1,test_tuned_pr_auc_macro,test_tuned_tn,test_tuned_fp,test_tuned_fn,test_tuned_tp,selected_metric,eps
0,undersampled,XGBoost,"{'clf__learning_rate': 0.2, 'clf__max_depth': ...",0.63349,0.28,0.678683,0.057308,0.066822,0.28,0.871703,...,0.967488,0.46692,0.967488,0.717204,1033,1086,1278,12660,macro_f1,0.01


In [48]:
cand_oof_macro, best_oof_macro, oof_pipe, oof_t, oof_splits = run_oof_80_20_pipeline(
    metric="macro_f1", use_gpu_for_xgb=False
)
best_oof_macro

Complete-case cohort: (160566, 31) positive rate: 0.868067
Train: (128452, 31) Test: (32114, 31)
Done: unbalanced | Logistic Regression | oof macro_f1 : 0.676102 | oof f1_0: 0.444115 | oof f1_1: 0.90809 | abs(f1_0-f1_1): 0.463974 | oof pr_auc_0: 0.41617 | oof pr_auc_1: 0.960966 | oof pr_auc_macro: 0.688568 | best_t: 0.74
Done: unbalanced | Decision Tree | oof macro_f1 : 0.638145 | oof f1_0: 0.380479 | oof f1_1: 0.895812 | abs(f1_0-f1_1): 0.515333 | oof pr_auc_0: 0.330435 | oof pr_auc_1: 0.938742 | oof pr_auc_macro: 0.634588 | best_t: 0.79
Done: unbalanced | Random Forest | oof macro_f1 : 0.645259 | oof f1_0: 0.401138 | oof f1_1: 0.889379 | abs(f1_0-f1_1): 0.48824 | oof pr_auc_0: 0.363255 | oof pr_auc_1: 0.953673 | oof pr_auc_macro: 0.658464 | best_t: 0.84
Done: unbalanced | XGBoost | oof macro_f1 : 0.695431 | oof f1_0: 0.472472 | oof f1_1: 0.91839 | abs(f1_0-f1_1): 0.445918 | oof pr_auc_0: 0.47247 | oof pr_auc_1: 0.968366 | oof pr_auc_macro: 0.720418 | best_t: 0.73
Done: weighted | Log

,workflow,model,params,oof_best_threshold,oof_best_metric_val,oof_threshold,oof_accuracy,oof_balanced_accuracy,oof_precision,oof_recall,...,test_tuned_roc_auc,test_tuned_pr_auc,test_tuned_pr_auc_0,test_tuned_pr_auc_1,test_tuned_pr_auc_macro,test_tuned_tn,test_tuned_fp,test_tuned_fn,test_tuned_tp,selected_metric
0,unbalanced,XGBoost,"{'clf__learning_rate': 0.1, 'clf__max_depth': ...",0.73,0.695431,0.73,0.858648,0.698009,0.920562,0.916228,...,0.828486,0.9675,0.456608,0.9675,0.712054,1965,2272,2396,25481,macro_f1


In [49]:
def pick_winner_between_tvt_and_oof(best_tvt_df, best_oof_df, eps=0.01):
    tvt = best_tvt_df.iloc[0]
    oof = best_oof_df.iloc[0]

    summary = pd.DataFrame([
        {
            "source": "TVT",
            "workflow": tvt["workflow"],
            "model": tvt["model"],
            "ref_macro_f1": float(tvt["valid_tuned_macro_f1"]),
            "test_macro_f1": float(tvt["test_tuned_macro_f1"]),
            "gap_to_test": float(abs(float(tvt["valid_tuned_macro_f1"]) - float(tvt["test_tuned_macro_f1"]))),
            "threshold": float(tvt["best_threshold_valid"]),
        },
        {
            "source": "OOF",
            "workflow": oof["workflow"],
            "model": oof["model"],
            "ref_macro_f1": float(oof["oof_macro_f1"]),
            "test_macro_f1": float(oof["test_tuned_macro_f1"]),
            "gap_to_test": float(abs(float(oof["oof_macro_f1"]) - float(oof["test_tuned_macro_f1"]))),
            "threshold": float(oof["oof_best_threshold"]),
        },
    ])

    best_ref = float(summary["ref_macro_f1"].max())
    pool = summary[summary["ref_macro_f1"] >= (best_ref - float(eps))].copy()

    winner_row = pool.sort_values(by=["gap_to_test", "ref_macro_f1"], ascending=[True, False]).iloc[0]
    return summary, winner_row

In [50]:
summary, winner_row = pick_winner_between_tvt_and_oof(best_macro, best_oof_macro, eps=0.01)
summary

,source,workflow,model,ref_macro_f1,test_macro_f1,gap_to_test,threshold
0,TVT,undersampled,XGBoost,0.678683,0.690487,0.011803,0.28
1,OOF,unbalanced,XGBoost,0.695431,0.686586,0.008845,0.73


In [51]:
winner_row

source                  OOF
workflow         unbalanced
model               XGBoost
ref_macro_f1       0.695431
test_macro_f1      0.686586
gap_to_test        0.008845
threshold              0.73
Name: 1, dtype: object

In [52]:
winner_source = str(winner_row["source"])
if winner_source == "OOF":
    winner_pipe = oof_pipe
    winner_t = float(oof_t)
else:
    winner_pipe = tvt_pipe
    winner_t = float(tvt_t)

print("Winner:", winner_row["source"], winner_row["workflow"], winner_row["model"])
print("Threshold:", winner_t)
print("Steps:", list(winner_pipe.named_steps.keys()))

Winner: OOF unbalanced XGBoost
Threshold: 0.73
Steps: ['prep', 'clf']


In [53]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer

def _get_feature_names_from_preprocessor(prep: ColumnTransformer):
    return prep.get_feature_names_out()

def extract_feature_importance_full(fitted_pipe) -> pd.DataFrame:
    prep = fitted_pipe.named_steps["prep"]
    clf = fitted_pipe.named_steps["clf"]

    if not hasattr(clf, "feature_importances_"):
        raise ValueError("This model does not expose feature_importances_. Use RF/DT/XGB.")

    feat_names = _get_feature_names_from_preprocessor(prep)
    imp = np.asarray(clf.feature_importances_, dtype=float)

    if len(feat_names) != len(imp):
        raise RuntimeError(f"Mismatch: {len(feat_names)} names vs {len(imp)} importances.")

    fi_all = (
        pd.DataFrame({"feature": feat_names, "importance": imp})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    return fi_all

# 1) FULL raw importances (all OHE columns + all numeric columns)
fi_all = extract_feature_importance_full(winner_pipe)


In [54]:
import pandas as pd
import numpy as np

def collapse_ohe_sum_single_model_using_lists(
    fi_long: pd.DataFrame,          # columns: ["feature","importance"]
    cat_feat: list,
    num_feat: list,
    top_n: int = 50
) -> pd.DataFrame:
    """
    Collapsing logic (NO REGEX):
      - strip prefix 'cat__'/'num__'/'remainder__' if present via split("__",1)
      - categorical OHE features collapse back to parent feature in cat_feat (longest-first match)
      - numeric features keep as-is
      - aggregate by SUM importance at parent level
    """

    cat_parents_sorted = sorted(cat_feat, key=len, reverse=True)

    def strip_prefix(s: str) -> str:
        return s.split("__", 1)[-1]

    def parent_from_cat_raw(raw: str) -> str:
        for parent in cat_parents_sorted:
            if raw == parent or raw.startswith(parent + "_"):
                return parent
        return raw.split("_", 1)[0] if "_" in raw else raw

    def parent_feature_name(encoded_name: str) -> str:
        if pd.isna(encoded_name):
            return encoded_name

        s = str(encoded_name)
        raw = strip_prefix(s)

        if s.startswith("num__") or raw in num_feat:
            return raw

        if s.startswith("cat__"):
            return parent_from_cat_raw(raw)

        # fallback resolution
        if raw in num_feat:
            return raw
        if raw in cat_feat:
            return raw

        return parent_from_cat_raw(raw)

    tmp = fi_long[["feature", "importance"]].dropna().copy()
    tmp["parent_feature"] = tmp["feature"].apply(parent_feature_name)

    collapsed = (
        tmp.groupby("parent_feature", as_index=False)["importance"]
           .sum()
           .sort_values("importance", ascending=False)
           .head(int(top_n))
           .reset_index(drop=True)
           .rename(columns={"parent_feature": "feature"})
    )

    return collapsed

In [55]:
X, y, meta = load_complete_case_dataset(
    data_path=DATA_PATH,
    cols_to_drop_path=COLS_TO_DROP_PATH,
    num_feat_path=NUM_FEAT_PATH,
    removed_features_path=REMOVED_FEATURES_PATH,
    target_col="EMSSceneTimeMin",
    key_col="PcrKey",
    y_col="scene_time_binary",
    threshold_minutes=10,
)

cat_feat = meta["cat_feat"]
num_feat = meta["num_feat"]


In [56]:
# 2) Collapse using ALL columns, then take top parents
collapsed_all = collapse_ohe_sum_single_model_using_lists(
    fi_long=fi_all,
    cat_feat=cat_feat,
    num_feat=num_feat,
    top_n=50
)

collapsed_all  # this is your intended top-50 parent features

,feature,importance
0,eProcedures_03_ID,0.250152
1,eScene_09_ID,0.128264
2,eSituation_11_ID,0.110455
3,eSituation_09_ID,0.078762
4,eProcedures_10_ID,0.057657
5,eDisposition_20_ID,0.042112
6,eResponse_24_ID,0.038794
7,USCensusDivision_ID,0.036569
8,eDisposition_17,0.026083
9,eVitals_20_ID,0.025560


## Feature Categories

### Clinical Features
Features directly determined by on-scene provider decisions — primary targets for counterfactual substitution.

| Feature | Description |
|---------|-------------|
| `eProcedures_03` | Procedure performed on the patient |
| `eSituation_09` | Primary symptom |
| `eSituation_11` | Provider's primary impression |
| `eProcedures_06` | Procedure successful (Boolean) |
| `eSituation_13` | Initial patient acuity |

### Operational Features
Driven by dispatch, protocols, and system configuration — controllable through policy.

| Feature | Description |
|---------|-------------|
| `eDisposition_20` | Reason for choosing destination |
| `eResponse_24` | Response mode techniques used |
| `eDisposition_17` | Transport mode from scene (emergent/non-emergent) |
| `eResponse_07` | Unit transport and equipment capability |
| `eScene_01` | First EMS unit on scene (Boolean) |
| `eDisposition_32` | Level of care provided per protocol |
| `eProcedures_10` | Type/level of EMS performing the procedure |

### Timing Features
Derived response time intervals — reflect system and scene efficiency.

| Feature | Description |
|---------|-------------|
| `EMSTotalCallTimeMin` | Total call time |
| `EMSSceneToPatientTimeMin` | Time from scene arrival to patient contact |
| `EMSTransportTimeMin` | Transport time |
| `EMSSceneResponseTimeMin` | Time from dispatch to scene arrival |
| `EMSChuteTimeMin` | Chute time (dispatch to unit response) |

### Patient Features
Patient physiology and demographics — fixed at time of call, used for subgrouping.

| Feature | Description |
|---------|-------------|
| `eVitals_20` | Glasgow Coma Score (verbal) |
| `eVitals_02` | Vitals obtained prior to this unit's care |
| `ePatient_14` | Race |
| `ePatient_15` | Age |
| `avg_eVitals_06` | Average systolic blood pressure |
| `avg_eVitals_10` | Average heart rate |
| `avg_eVitals_12` | Average pulse oximetry (SpO2) |
| `avg_eVitals_14` | Average respiratory rate |
| `avg_eVitals_18` | Average blood glucose |

### Geographic Features
Location and system-level context — used for subgrouping.

| Feature | Description |
|---------|-------------|
| `eScene_09` | Type of location |
| `USCensusDivision` | US Census division |
| `USCensusRegion` | US Census region |
| `Urbanicity` | Urban/suburban/rural classification |
| `NasemsoRegion` | NASEMSO region |


In [57]:
collapsed_all["importance"].sum()

np.float64(1.000000082407496)